In [ ]:
# ===================================================================================
# WEB EXPERIENCE MIGRATION (V16.7 - STRICT SHARING + GROUPS + THUMBNAILS)
# - FIX: Solves "Item Not Found" by checking migration_history.csv for IDs.
# - UPDATE: Strict Sharing Mirroring (Groups + Org + Public/Private).
# - UPDATE: Shares BEFORE ownership transfer to avoid privilege errors.
# - UPDATE: Copies Thumbnails & Resources (Images) via REST.
# ===================================================================================

import json
import copy
import time
import csv
import os
import shutil
import re
import datetime
import urllib3
import requests
import sys
import warnings
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

# --- SUPPRESS WARNINGS --------------------------------------------------------
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC OVERRIDES ---
THROTTLE_SECONDS = 15

# LIST YOUR EXPERIENCE IDs HERE
# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_4_experiences.json")
if os.path.exists(_sidecar_path):
    import json as _json
    EXPERIENCE_IDS_TO_MIGRATE = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(EXPERIENCE_IDS_TO_MIGRATE)} Experience IDs from sidecar.")
else:
    EXPERIENCE_IDS_TO_MIGRATE = [
        # Example Source ID
        # "04c534a6ab7b422891ee95a540d226e2",
        # "e054257e2da8448e98961b6a0c950f23",


    ]

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    
    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)
    
    print(f"   > Source: {source_gis.url}")
    print(f"   > Target: {target_gis.users.me.username}")
except Exception as e:
    print(f"\n❌ CRITICAL CONNECTION FAILURE: {e}")
    sys.exit(1)

# =============================================================================
# --- LEDGER LOADER ------------------------------------------------------------
# =============================================================================
STATS = {"scanned": 0, "created": 0, "failed": 0, "skipped_log": 0}
ALREADY_MIGRATED_IDS = set()
ID_MAP = {} # Memory for SourceID -> TargetID lookups

if os.path.exists(LOG_FILE):
    try:
        df = pd.read_csv(LOG_FILE)
        # Load Already Migrated Log
        if 'SourceID' in df.columns: 
            ALREADY_MIGRATED_IDS = set(df['SourceID'].astype(str).str.strip())
        
        # Load Mapping for Swapping (The critical part)
        if "SourceID" in df.columns and "TargetID" in df.columns:
            for _, row in df.iterrows():
                s_id = str(row["SourceID"]).strip()
                t_id = str(row["TargetID"]).strip()
                if s_id and t_id and s_id != "nan" and t_id != "nan":
                    ID_MAP[s_id] = t_id
        print(f"✅ Loaded Ledger: {len(ID_MAP)} ID mappings found.")
    except Exception as e:
        print(f"⚠️ Failed to load CSV: {e}")
else:
    # Initialize if missing
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except: pass

def log_migration(source_id, target_id, title):
    try:
        with open(LOG_FILE, 'a', newline='') as f:
            csv.writer(f).writerow([source_id, "N/A", target_id, title, datetime.datetime.now(), "Web Experience"])
            ALREADY_MIGRATED_IDS.add(str(source_id))
    except: pass

# =============================================================================
# --- HELPER: REST API & FOLDER UTILS ------------------------------------------
# =============================================================================
def remove_resource_via_rest(item_id, resource_path):
    url = f"{TARGET_URL}/sharing/rest/content/users/{target_gis.users.me.username}/items/{item_id}/removeResources"
    data = {"f": "json", "token": TARGET_TOKEN, "resource": resource_path}
    try: session.post(url, data=data, verify=False)
    except: pass

def add_resource_via_rest(item_id, local_file, file_name, prefix=None, overwrite=True):
    if overwrite and prefix:
        remove_resource_via_rest(item_id, f"{prefix}/{file_name}")

    url = f"{TARGET_URL}/sharing/rest/content/users/{target_gis.users.me.username}/items/{item_id}/addResources"
    with open(local_file, "rb") as f:
        data = {"f": "json", "token": TARGET_TOKEN, "fileName": file_name, "access": "inherit"}
        if prefix: data["resourcesPrefix"] = prefix 
        r = session.post(url, data=data, files={"file": f}, verify=False)
    return r.json()

def ensure_target_folder_exists(username, folder_name):
    """Ensure a folder exists for a specific user in Target."""
    try:
        user = target_gis.users.get(username)
        if not user: return False
        
        # Check existing folders (Handles Folder Objects AND Dicts)
        existing_folders = []
        for f in user.folders:
            if hasattr(f, 'title'):
                existing_folders.append(f.title)
            elif isinstance(f, dict):
                existing_folders.append(f.get('title'))
        
        if folder_name in existing_folders: return True
        
        # Create if missing
        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except Exception as e: 
        print(f"      ⚠ Folder creation error: {e}")
        return False

def assign_correct_owner_and_folder(source_item, target_item):
    try:
        target_owner = DEFAULT_OWNER
        target_folder = DEFAULT_FOLDER
        
        if target_gis.users.get(source_item.owner):
            target_owner = source_item.owner
            src_folder = None
            try:
                u = source_gis.users.get(source_item.owner)
                for f in u.folders:
                    if f['id'] == source_item.ownerFolder: src_folder = f['title']
            except: pass

            if src_folder:
                ensure_target_folder_exists(target_owner, src_folder)
                target_folder = src_folder
            else:
                target_folder = None

        print(f"      📂 Moving to: Owner={target_owner}, Folder={target_folder}")
        try:
            target_item.reassign_to(target_owner, target_folder=target_folder)
        except:
            if target_owner != DEFAULT_OWNER:
                print(f"      🔄 Fallback: Reassigning to '{DEFAULT_OWNER}'...")
                target_item.reassign_to(DEFAULT_OWNER, target_folder=DEFAULT_FOLDER)
    except Exception as e:
        print(f"      ⚠ Reassign Error: {e}")

# =============================================================================
# --- HELPER: THUMBNAIL & SHARING ----------------------------------------------
# =============================================================================
def copy_thumbnail(source_item, target_item):
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = "thumbnails_temp"
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_item.download_thumbnail(save_folder=temp_dir)
        if thumb_path: target_item.update(thumbnail=thumb_path)
    except: pass

def mirror_source_sharing(source_item, target_item):
    """
    1. Checks Source Sharing (Public/Org/Private).
    2. STRICTLY applies the same level to Target (No forcing Org).
    3. Maps Source Groups -> Target Groups (by exact Title).
    """
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")
        
        # 1. Access Level
        is_public = False
        is_org = False
        
        if source_item.access == 'public':
            is_public = True
            is_org = True 
        elif source_item.access == 'org':
            is_org = True
        # Else: Private (Both remain False)
            
        # 2. Map Groups (Robust retrieval)
        source_groups = []
        try:
            # Try getting via 'sharing.groups'
            raw_groups = source_item.sharing.groups
            # Explicitly check if it's a list. If not, fail to except block.
            if isinstance(raw_groups, list):
                source_groups = raw_groups
            else:
                raise ValueError("Not a list")
        except:
            # Fallback to 'shared_with'
            raw_shared = source_item.shared_with
            if isinstance(raw_shared, dict) and 'groups' in raw_shared:
                source_groups = raw_shared['groups']
            elif isinstance(raw_shared, list):
                source_groups = raw_shared

        target_group_ids = []
        
        if source_groups:
            print(f"         - Found {len(source_groups)} source groups. Mapping...")
            for sg in source_groups:
                # Handle Group Object vs String Name
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)

                # Search for group with exact title in Target
                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                
                # Filter for exact match
                matched_group = next((g for g in found_groups if g.title == sg_title), None)
                
                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")
        
        # 3. Apply
        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")

    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")

# =============================================================================
# --- HELPER: DEEP ID SWAP (LEDGER PRIORITY) -----------------------------------
# =============================================================================
SEARCH_CACHE = {}

def deep_scan_and_swap_ids(json_str):
    """
    1. Scans JSON for 32-char IDs.
    2. Checks CSV Ledger FIRST (Fast & Accurate).
    3. Checks Portal Search SECOND (Fallback).
    4. Swaps IDs in the JSON string.
    """
    print("   🔍 Deep Scanning for embedded IDs...")
    
    # Regex for 32-character Hex String
    potential_ids = set(re.findall(r'\b[0-9a-f]{32}\b', json_str))
    
    swaps_made = 0
    new_json_str = json_str
    
    for old_id in potential_ids:
        new_id = None
        
        # 1. CHECK LEDGER (Priority)
        if old_id in ID_MAP:
            new_id = ID_MAP[old_id]
            
        # 2. CHECK SEARCH CACHE
        elif old_id in SEARCH_CACHE:
            new_id = SEARCH_CACHE[old_id]
            
        # 3. CHECK PORTAL SEARCH (Fallback)
        else:
            try:
                res = target_gis.content.search(f'tags:"SourceID:{old_id}"', max_items=1)
                if res:
                    new_id = res[0].id
                    SEARCH_CACHE[old_id] = new_id
                else:
                    SEARCH_CACHE[old_id] = None
            except:
                SEARCH_CACHE[old_id] = None

        # 4. PERFORM SWAP
        if new_id:
            if old_id != new_id: 
                new_json_str = new_json_str.replace(old_id, new_id)
                swaps_made += 1
                if old_id == "1132c531509e4000a293fef3d39db052":
                    print(f"      🎯 FIXED MISSING DASHBOARD LINK: {old_id} -> {new_id}")

    if swaps_made > 0:
        print(f"      ✅ Swapped {swaps_made} embedded references.")
    else:
        print("      ℹ️ No embedded IDs found/swapped.")
        
    return new_json_str

# =============================================================================
# --- HELPER: RESOURCE COPY ----------------------------------------------------
# =============================================================================
def copy_resources_manual(src_item, tgt_item):
    print("   📂 Copying Assets (REST Mode)...")
    temp_dir = f"temp_res_{src_item.id}"
    if not os.path.exists(temp_dir): os.makedirs(temp_dir)
    
    count = 0
    try:
        url = f"{source_gis.url}/sharing/rest/content/items/{src_item.id}/resources"
        res = requests.get(url, params={'f': 'json', 'token': SOURCE_TOKEN}, verify=False)
        resources = res.json().get('resources', [])

        for r_obj in resources:
            r_path = r_obj['resource'] if isinstance(r_obj, dict) else r_obj
            if "config/config.json" in r_path or r_path == "config.json": continue 
            
            dl_url = f"{source_gis.url}/sharing/rest/content/items/{src_item.id}/resources/{r_path}"
            r_res = requests.get(dl_url, params={'token': SOURCE_TOKEN}, stream=True, verify=False)
            
            if r_res.status_code == 200:
                clean_path = r_path.replace("\\", "/")
                local_path = os.path.join(temp_dir, clean_path.replace("/", os.sep))
                os.makedirs(os.path.dirname(local_path), exist_ok=True)
                with open(local_path, 'wb') as f:
                    for chunk in r_res.iter_content(chunk_size=8192): f.write(chunk)
                
                folder_part, file_part = os.path.split(clean_path)
                prefix = folder_part if folder_part else None
                try:
                    add_resource_via_rest(tgt_item.id, local_path, file_part, prefix=prefix, overwrite=True)
                    count += 1
                except: pass

        print(f"      ✅ Copied {count} assets.")
    except Exception as e:
        print(f"      ⚠ Resource Copy Failed: {e}")
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print(f"Starting Web Experience Migration (V16.7 - Strict Sharing)...")
start_time = time.time()

for app_id in EXPERIENCE_IDS_TO_MIGRATE:
    try:
        STATS["scanned"] += 1
        
        # We allow re-processing to fix broken links
        if str(app_id) in ALREADY_MIGRATED_IDS:
             pass 

        src_item = source_gis.content.get(app_id)
        if not src_item:
            print(f"❌ Source {app_id} not found.")
            STATS["failed"] += 1
            continue

        print(f"\nProcessing: {src_item.title}...")
        
        data = src_item.get_data()
        if not data: continue
        if "isTemplate" in data: data["isTemplate"] = False

        target_title = f"{src_item.title} (Migrated)" if APPEND_MIGRATED else src_item.title
        
        existing_list = target_gis.content.search(f'title:"{target_title}"', max_items=1)
        if existing_list:
            print(f"   ⚠️ Item exists. Updating config to fix links...")
            new_app = existing_list[0]
        else:
            final_tags = list(src_item.tags) + ["Migrated", f"SourceID:{app_id}"]
            type_keywords = list(src_item.typeKeywords) + ["Experience Builder", "Web Experience"]
            item_props = {
                "type": "Web Experience",
                "title": target_title,
                "snippet": src_item.snippet,
                "description": src_item.description,
                "tags": list(set(final_tags)),
                "typeKeywords": list(set(type_keywords)),
                "text": "{}" 
            }
            new_app = target_gis.content.add(item_props, folder=DEFAULT_FOLDER)

        if new_app:
            print("   💉 Patching Config...")
            json_str = json.dumps(data)
            
            # 1. Standard Self-ID Swap
            patched_str = json_str.replace(app_id, new_app.id) 
            
            # 2. URL Domain Swap
            clean_src = SOURCE_URL.rstrip("/")
            clean_tgt = TARGET_URL.rstrip("/")
            if clean_src in patched_str:
                patched_str = patched_str.replace(clean_src, clean_tgt)

            # 3. DEEP ID SWAP (LEDGER + SEARCH)
            patched_str = deep_scan_and_swap_ids(patched_str)

            # Update URL and Text
            app_url = f"{clean_tgt}/apps/experiencebuilder/experience/?id={new_app.id}"
            new_app.update(item_properties={"url": app_url, "text": patched_str})

            # Upload Config File (Critical for ExB)
            try:
                temp_conf = f"temp_conf_{new_app.id}"
                os.makedirs(temp_conf, exist_ok=True)
                config_path = os.path.join(temp_conf, "config.json")
                with open(config_path, "w", encoding="utf-8") as f: f.write(patched_str)
                add_resource_via_rest(new_app.id, config_path, "config.json", prefix="config", overwrite=True)
                print("      ✅ Config Uploaded.")
            finally:
                shutil.rmtree(temp_conf, ignore_errors=True)

            # Copy Resources (Images)
            copy_resources_manual(src_item, new_app) 
            
            # Copy Thumbnail
            copy_thumbnail(src_item, new_app)

            # --- STRICT SHARING MIRROR ---
            # CALLED BEFORE REASSIGNING OWNER TO AVOID PRIVILEGE ERRORS
            mirror_source_sharing(src_item, new_app)

            # Assign Owner & Folder
            assign_correct_owner_and_folder(src_item, new_app)

            log_migration(app_id, new_app.id, new_app.title)
            STATS["created"] += 1
            print(f"🚀 SUCCESS: Processed '{new_app.title}'")
        
        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ Failed: {e}")
        STATS["failed"] += 1

# =============================================================================
# --- FINAL REPORT -------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "="*60)
print("      WEB EXPERIENCE MIGRATION REPORT (V16.7)  ")
print("="*60)
print(f" ⏱️  Total Duration:            {duration_str}")
print("-" * 60)
print(f" 📱 Apps Scanned:             {STATS['scanned']}")
print(f" ⏭️  Skipped (CSV):             {STATS['skipped_log']}")
print(f" ✅ Apps Created:             {STATS['created']}")
print(f" ❌ Failures:                 {STATS['failed']}")
print("="*60)